# 01 — Corpus EDA

Inventory of the Atlas Industries corpus: file formats, languages, sizes, and
a spot-check that Arabic text extracts correctly (FR-B3).

In [1]:
import sys
from pathlib import Path

# Works whether Jupyter was started from the repo root or from notebooks/:
# walk up until we find the package (starting `from src...` from the wrong
# cwd is the classic notebook failure).
here = Path.cwd()
root = next(p for p in [here, *here.parents] if (p / "src" / "settings.py").exists())
sys.path.insert(0, str(root))
print("project root:", root)

from src.chunking import detect_language
from src.loaders import SUPPORTED_EXTENSIONS, load_document
from src.settings import settings

print("docs dir:", settings.docs_dir)

project root: /home/nourz/Desktop/Atlas_sprintsProject
docs dir: /home/nourz/Desktop/Atlas_sprintsProject/data/docs


## Format × language inventory

In [2]:
rows = []
for domain in settings.domains:
    for path in sorted((settings.docs_dir / domain).iterdir()):
        if path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue
        text = load_document(path)
        rows.append(
            {
                "domain": domain,
                "file": path.name,
                "format": path.suffix.lower().lstrip("."),
                "language": detect_language(text),
                "chars": len(text),
            }
        )

import pandas as pd

df = pd.DataFrame(rows)
df

,domain,file,format,language,chars
0,hr,HR-001-vacation-policy.md,md,en,2115
1,hr,HR-002-remote-work-policy.pdf,pdf,en,2648
2,hr,HR-003-onboarding-handbook.docx,docx,en,3428
3,hr,HR-004-code-of-conduct.md,md,en,2911
4,hr,HR-005-performance-review.pdf,pdf,en,2261
5,hr,HR-006-leave-types-guide.docx,docx,en,3042
6,hr,HR-007-سياسة-الإجازات-المرضية.md,md,ar,1652
7,hr,HR-008-دليل-التوظيف.pdf,pdf,ar,1929
8,hr,HR-009-ساعات-العمل-المرنة.docx,docx,ar,2074
9,hr,HR-010-grievance-procedure-bilingual.md,md,bilingual,2528


In [3]:
print("Documents per domain:")
print(df["domain"].value_counts().to_string())
print("\nDocuments per format (brief expects ~4 md / 3 pdf / 3 docx per domain):")
print(df.groupby(["domain", "format"]).size().to_string())
print("\nDocuments per language (brief expects ~60% en / 30% ar / 10% bilingual):")
print(df["language"].value_counts(normalize=True).round(2).to_string())

Documents per domain:
domain
hr         10
it         10
finance    10

Documents per format (brief expects ~4 md / 3 pdf / 3 docx per domain):
domain   format
finance  docx      3
         md        4
         pdf       3
hr       docx      3
         md        4
         pdf       3
it       docx      3
         md        4
         pdf       3

Documents per language (brief expects ~60% en / 30% ar / 10% bilingual):
language
en           0.6
ar           0.3
bilingual    0.1


## FR-B3 — Arabic extraction sanity check

Print raw excerpts from one Arabic PDF and one Arabic DOCX. Scrambled glyphs or
character-reversed lines mean the PDF loader is wrong (see `src/loaders.py` for
why `pypdf` was chosen over `pdfplumber`).

In [4]:
arabic_samples = {
    "PDF": settings.docs_dir / "finance" / "FIN-008-بدل-المواصلات.pdf",
    "DOCX": settings.docs_dir / "finance" / "FIN-009-عشاء-العملاء.docx",
    "MD": settings.docs_dir / "hr" / "HR-007-سياسة-الإجازات-المرضية.md",
}

for kind, path in arabic_samples.items():
    text = load_document(path)
    print(f"=== {kind}: {path.name} ({len(text)} chars) ===")
    print(text[:400].replace("\n", " | "))
    print()

=== PDF: FIN-008-بدل-المواصلات.pdf (1708 chars) ===
شركة أطلس الصناعية — بدل المواصلات اليومي | رقم السياسة:FIN-TRA-008 الجهة المسؤولة الإدارة المالية تاريخ السريان: 1 يناير   | 2025 الإصدار 2.0 | 1. الغرض | تحدد هذه السياسة بدل المواصلات الشهري للموظفين، وآلية الصرف، والاستثناءات الخاصة  | برحلات العمل. | 2. الاستحقاق الشهري | الفئة بدل المواصلات الشهري | الموظفون (مستوى Associate) 1,200 جنيه مصري | كبار الموظفين (مستوى Senior) 1,800 جنيه مصري | المدراء 2,500 جنيه مص

=== DOCX: FIN-009-عشاء-العملاء.docx (2412 chars) ===
شركة أطلس الصناعية — سياسة استضافة العملاء والعشاء التجاري | رقم السياسة: FIN-CLI-009 الجهة المسؤولة: الإدارة المالية تاريخ السريان: 1 يناير 2025 الإصدار: 2.4 | 1. الغرض | تحدد هذه السياسة قواعد وضوابط استرداد نفقات استضافة العملاء (الإفطار، الغداء، العشاء، الفعاليات) من قبل موظفي شركة أطلس الصناعية. | 2. النطاق | تنطبق هذه السياسة على: - وجبات العمل مع عملاء حاليين أو محتملين. - الفعاليات الترفيهية المرتبط

=== MD: HR-007-سياسة-الإجازات-المرضية.md (1652 chars) ===
# ش

## Chunking profile

How many chunks does the current `CHUNK_SIZE` / `CHUNK_OVERLAP` produce?
On a 30-doc corpus this drives retrieval ranking sensitivity (known concern).

In [5]:
from src.ingest import build_chunks

chunks = build_chunks()
print(f"{len(chunks)} chunks from {len(rows)} documents")
print("by domain:", pd.Series([c.domain for c in chunks]).value_counts().to_dict())
print("by language:", pd.Series([c.language for c in chunks]).value_counts().to_dict())
print("avg chunk chars:", int(pd.Series([len(c.text) for c in chunks]).mean()))
print("\nArabic filenames preserved through chunking:")
for c in chunks:
    if any("\u0600" <= ch <= "\u06ff" for ch in c.source_filename):
        print(" ", c.chunk_id, "->", c.source_filename)
        break

122 chunks from 30 documents
by domain: {'it': 42, 'finance': 42, 'hr': 38}
by language: {'en': 79, 'ar': 29, 'bilingual': 14}
avg chunk chars: 755

Arabic filenames preserved through chunking:
  HR-007-سياسة-الإجازات-المرضية::0 -> HR-007-سياسة-الإجازات-المرضية.md
